# Song Popularity Prediction via Random Forest (regression)

---

## Question & Background

### Research Question

**Which audio features best predict a song's popularity score on Spotify, and can we build a reliable decision tree model to estimate that score from measurable acoustic properties alone?**

Specifically, we examine the following features: song duration, acousticness, danceability, energy, instrumentalness, key, liveness, loudness, speechiness, tempo, time signature, and audio valence — all in relation to a popularity score ranging from 0 to 100.

---

### Background & Motivation

- Music recommendation and discovery has become one of the most commercially significant applications of machine learning in the modern era. Streaming platforms like Spotify serve over 600 million users and rely heavily on algorithmic curation to surface content. At the heart of this system is a concept deceptively simple in name but complex in nature: **song popularity**. Understanding *what makes a song popular* has direct implications for artists, producers, and record labels who want to understand how their music will perform before it is ever released.

- Spotify calculates a song's popularity score (0–100) primarily based on the **total number of streams a track has received, weighted toward more recent plays**. This means the score is dynamic and reflects current listener engagement rather than all-time historical performance. While streaming volume is the core signal, Spotify's algorithm also factors in how often a song is saved or added to playlists, giving the metric a broader sense of cultural resonance beyond passive listening.

- Our dataset, sourced from Kaggle via the Spotify Web API, contains **13,070 songs** with 13 quantitative audio features per track. These features are extracted algorithmically by Spotify's audio analysis engine and represent low-level acoustic and structural characteristics of each song — from how "danceable" a beat is, to how much of the audio is estimated to be a live performance. 

---

### Why This Question Matters

We chose this question because it sits at the intersection of music, data science, and real-world business value. Predicting popularity from audio features would allow people to make data-informed decisions about song selection, production choices, and promotional strategy — all *before* a release goes live. From a modeling standpoint, this is an interesting regression challenge because popularity is influenced by many factors *outside* our feature set (marketing budgets, artist fame, social media trends), which means our model will face real limitations. We believe that honestly confronting these limitations — and understanding how much variance in popularity *can* be explained by audio features alone — is itself a meaningful and instructive finding.

## Features in the Dataset - 
- **Acousticness** (float): A confidence measure from 0.0 to 1.0 of whether the track is acoustic. 1.0 represents high confidence the track is acoustic.
  
- **Danceability** (float): describes how suitable a track is for dancing based on a combination of musical elements including tempo, rhythm stability, beat strength, and overall regularity. A value of 0.0 is least danceable and 1.0 is most danceable.

- **Duration_ms** (int): The duration of the track in milliseconds.

- **Energy** (float): Energy is a measure from 0.0 to 1.0 and represents a perceptual measure of intensity and activity. Typically, energetic tracks feel fast, loud, and noisy.

- **Instrumentalness** (float): Predicts whether a track contains no vocals. "Ooh" and "aah" sounds are treated as instrumental in this context. Rap or spoken word tracks are clearly "vocal". The closer the instrumentalness value is to 1.0, the greater likelihood the track contains no vocal content.
- **Key** (int): The key the track is in. Integers map to pitches using standard Pitch Class notation. E.g. 0 = C, 1 = C♯/D♭, 2 = D, and so on. If no key was detected, the value is -1.
- **Liveness** (float): Detects the presence of an audience in the recording. Higher liveness values represent an increased probability that the track was performed live.
- **Loudness** (float): The overall loudness of a track in decibels (dB). Loudness values are averaged across the entire track and are useful for comparing relative loudness of tracks.
- **Mode** (int): indicates the modality (major or minor) of a track, the type of scale from which its melodic content is derived. Major is represented by 1 and minor is 0.
- **Speechiness** (float): detects the presence of spoken words in a track. The more exclusively speech-like the recording (e.g. talk show, audio book, poetry), the closer to 1.0 the attribute value. 
- **Tempo** (int): The overall estimated tempo of a track in beats per minute (BPM). In musical terminology, tempo is the speed or pace of a given piece and derives directly from the average beat duration.
- **Time_signature** (int): An estimated time signature. The time signature (meter) is a notational convention to specify how many beats are in each bar (or measure). 





### Import Data and Packages. 

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error, r2_score


In [ ]:
# importing the data
data = pd.read_csv('song_data.csv')
data.head()

,song_name,song_popularity,song_duration_ms,acousticness,danceability,energy,instrumentalness,key,liveness,loudness,audio_mode,speechiness,tempo,time_signature,audio_valence
0,Boulevard of Broken Dreams,73,262333,0.005520,0.496,0.682,0.000029,8,0.0589,-4.095,1,0.0294,167.060,4,0.474
1,In The End,66,216933,0.010300,0.542,0.853,0.000000,3,0.1080,-6.407,0,0.0498,105.256,4,0.370
2,Seven Nation Army,76,231733,0.008170,0.737,0.463,0.447000,0,0.2550,-7.828,1,0.0792,123.881,4,0.324
3,By The Way,74,216933,0.026400,0.451,0.970,0.003550,0,0.1020,-4.938,1,0.1070,122.444,4,0.198
4,How You Remind Me,56,223826,0.000954,0.447,0.766,0.000000,10,0.1130,-5.065,1,0.0313,172.011,4,0.574


In [13]:
# EDA and data cleaning
data.info()
data.describe()
data.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 18835 entries, 0 to 18834
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   song_name         18835 non-null  str    
 1   song_popularity   18835 non-null  int64  
 2   song_duration_ms  18835 non-null  int64  
 3   acousticness      18835 non-null  float64
 4   danceability      18835 non-null  float64
 5   energy            18835 non-null  float64
 6   instrumentalness  18835 non-null  float64
 7   key               18835 non-null  int64  
 8   liveness          18835 non-null  float64
 9   loudness          18835 non-null  float64
 10  audio_mode        18835 non-null  int64  
 11  speechiness       18835 non-null  float64
 12  tempo             18835 non-null  float64
 13  time_signature    18835 non-null  int64  
 14  audio_valence     18835 non-null  float64
dtypes: float64(9), int64(5), str(1)
memory usage: 2.2 MB


song_name           0
song_popularity     0
song_duration_ms    0
acousticness        0
danceability        0
energy              0
instrumentalness    0
key                 0
liveness            0
loudness            0
audio_mode          0
speechiness         0
tempo               0
time_signature      0
audio_valence       0
dtype: int64